<a href="https://colab.research.google.com/github/HariNiveditha/RideDemandForecast/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RideDemandForecast"


Mounted at /content/drive


In [3]:
import pandas as pd

april = pd.read_parquet(f"{base_path}/hourly_demand_04.parquet")
may = pd.read_parquet(f"{base_path}/hourly_demand_05.parquet")
june = pd.read_parquet(f"{base_path}/hourly_demand_06.parquet")

trip_df = pd.concat(
    [april, may, june],
    ignore_index=True
)

print(trip_df.shape)
print(trip_df.columns.tolist())

(550297, 3)
['pickup_hour', 'PULocationID', 'ride_count']


In [4]:
# ==========================================
# Load cleaned weather data
# ==========================================

weather_df = pd.read_parquet(
    f"{base_path}/weather_cleaned.parquet"
)

print("Weather shape:", weather_df.shape)
print("Weather columns:")
print(weather_df.columns.tolist())

print("\nWeather preview:")
print(weather_df.head())

Weather shape: (365, 19)
Weather columns:
['datetime', 'temp', 'feelslike', 'humidity', 'precip', 'precipprob', 'snow', 'windgust', 'windspeed', 'sealevelpressure', 'cloudcover', 'visibility', 'day', 'month', 'dayofweek', 'weekofyear', 'is_weekend', 'quarter', 'dayofyear']

Weather preview:
    datetime  temp  feelslike  humidity  precip  precipprob  snow  windgust  \
0 2021-01-01   2.5       -0.2      67.8   15.33         100   0.0      42.5   
1 2021-01-02   5.8        3.6      74.0    2.38         100   1.9      54.6   
2 2021-01-03   2.5       -1.6      80.7    5.09         100   1.2      42.2   
3 2021-01-04   3.6        1.1      76.6    0.84         100   0.5      42.5   
4 2021-01-05   3.8        1.3      68.7    0.00           0   0.0      31.7   

   windspeed  sealevelpressure  cloudcover  visibility  day  month  dayofweek  \
0       15.5            1028.9        50.6        14.0    1      1          4   
1       25.5            1012.4        63.9        12.2    2      1     

In [5]:
zone_df = pd.read_csv(
    f"{base_path}/taxi_zone_lookup.csv"
)

print("Zone shape:", zone_df.shape)
print(zone_df.columns.tolist())

Zone shape: (265, 4)
['LocationID', 'Borough', 'Zone', 'service_zone']


In [6]:
ride_weather_df = trip_df.merge(
    weather_df,
    left_on="pickup_hour",
    right_on="datetime",
    how="left"
)

print("ride_weather_df shape:", ride_weather_df.shape)
print(ride_weather_df.head())

ride_weather_df shape: (550297, 22)
  pickup_hour  PULocationID  ride_count   datetime  temp  feelslike  humidity  \
0  2021-04-01             3          26 2021-04-01   8.1        5.4      62.5   
1  2021-04-01             4          24 2021-04-01   8.1        5.4      62.5   
2  2021-04-01             6           5 2021-04-01   8.1        5.4      62.5   
3  2021-04-01             7          83 2021-04-01   8.1        5.4      62.5   
4  2021-04-01             9           6 2021-04-01   8.1        5.4      62.5   

   precip  precipprob  snow  ...  sealevelpressure  cloudcover  visibility  \
0    1.99       100.0   0.0  ...            1010.3        94.7        14.6   
1    1.99       100.0   0.0  ...            1010.3        94.7        14.6   
2    1.99       100.0   0.0  ...            1010.3        94.7        14.6   
3    1.99       100.0   0.0  ...            1010.3        94.7        14.6   
4    1.99       100.0   0.0  ...            1010.3        94.7        14.6   

   day  

In [7]:
combined_df = ride_weather_df.merge(
    zone_df,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

print("combined_df shape:", combined_df.shape)
print(combined_df.head())

combined_df shape: (550297, 26)
  pickup_hour  PULocationID  ride_count   datetime  temp  feelslike  humidity  \
0  2021-04-01             3          26 2021-04-01   8.1        5.4      62.5   
1  2021-04-01             4          24 2021-04-01   8.1        5.4      62.5   
2  2021-04-01             6           5 2021-04-01   8.1        5.4      62.5   
3  2021-04-01             7          83 2021-04-01   8.1        5.4      62.5   
4  2021-04-01             9           6 2021-04-01   8.1        5.4      62.5   

   precip  precipprob  snow  ...  month  dayofweek  weekofyear  is_weekend  \
0    1.99       100.0   0.0  ...    4.0        3.0        13.0         0.0   
1    1.99       100.0   0.0  ...    4.0        3.0        13.0         0.0   
2    1.99       100.0   0.0  ...    4.0        3.0        13.0         0.0   
3    1.99       100.0   0.0  ...    4.0        3.0        13.0         0.0   
4    1.99       100.0   0.0  ...    4.0        3.0        13.0         0.0   

   quarter  

In [8]:
# Check unmatched pickup location IDs

missing_locations = combined_df.loc[
    combined_df["Zone"].isna(),
    "PULocationID"
].unique()

print("Number of unmatched LocationIDs:", len(missing_locations))
print("Unmatched LocationIDs:")
print(sorted(missing_locations))

Number of unmatched LocationIDs: 1
Unmatched LocationIDs:
[np.int64(265)]


In [10]:
# ==========================================
# Create final modeling dataframe
# ==========================================

model_df = combined_df.copy()

# Create pickup_date directly from pickup_hour
model_df["pickup_date"] = pd.to_datetime(
    model_df["pickup_hour"]
).dt.normalize()

# Sort chronologically
model_df = model_df.sort_values(
    ["pickup_date", "pickup_hour", "PULocationID"]
).reset_index(drop=True)

print("Date range:")
print(
    model_df["pickup_date"].min(),
    "to",
    model_df["pickup_date"].max()
)

print("\nModel dataframe shape:")
print(model_df.shape)

print("\nMissing pickup_date values:")
print(model_df["pickup_date"].isna().sum())

Date range:
2021-04-01 00:00:00 to 2021-06-30 00:00:00

Model dataframe shape:
(550297, 27)

Missing pickup_date values:
0


In [11]:
features = [
    "pickup_hour",
    "PULocationID",
    "temp",
    "feelslike",
    "humidity",
    "precip",
    "precipprob",
    "snow",
    "windgust",
    "windspeed",
    "sealevelpressure",
    "cloudcover",
    "visibility",
    "day",
    "month",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "quarter",
    "dayofyear"
]

X = model_df[features]
y = model_df["ride_count"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

X shape: (550297, 20)
y shape: (550297,)
Missing values in X: 9493668
Missing values in y: 0


In [12]:
# ==========================================
# Check missing values feature-by-feature
# ==========================================

missing = X.isna().sum()

print("Missing values by feature:")
print(missing[missing > 0].sort_values(ascending=False))

print("\nTotal missing values:", missing.sum())

Missing values by feature:
temp                527426
feelslike           527426
humidity            527426
precip              527426
precipprob          527426
snow                527426
windgust            527426
windspeed           527426
sealevelpressure    527426
cloudcover          527426
visibility          527426
day                 527426
month               527426
dayofweek           527426
weekofyear          527426
is_weekend          527426
quarter             527426
dayofyear           527426
dtype: int64

Total missing values: 9493668


In [14]:
print("trip_df columns:")
print(trip_df.columns.tolist())

print("\nweather_df columns:")
print(weather_df.columns.tolist())

print("\ntrip_df pickup_hour sample:")
print(trip_df["pickup_hour"].head())

print("\nweather_df datetime sample:")
print(weather_df["datetime"].head())

trip_df columns:
['pickup_hour', 'PULocationID', 'ride_count']

weather_df columns:
['datetime', 'temp', 'feelslike', 'humidity', 'precip', 'precipprob', 'snow', 'windgust', 'windspeed', 'sealevelpressure', 'cloudcover', 'visibility', 'day', 'month', 'dayofweek', 'weekofyear', 'is_weekend', 'quarter', 'dayofyear']

trip_df pickup_hour sample:
0   2021-04-01
1   2021-04-01
2   2021-04-01
3   2021-04-01
4   2021-04-01
Name: pickup_hour, dtype: datetime64[us]

weather_df datetime sample:
0   2021-01-01
1   2021-01-02
2   2021-01-03
3   2021-01-04
4   2021-01-05
Name: datetime, dtype: datetime64[ns]


In [15]:
# ==========================================
# Recreate ride + weather merge correctly
# ==========================================

trip_df["pickup_hour"] = pd.to_datetime(trip_df["pickup_hour"]).dt.normalize()
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"]).dt.normalize()

ride_weather_df = trip_df.merge(
    weather_df,
    left_on="pickup_hour",
    right_on="datetime",
    how="left"
)

print("Shape:", ride_weather_df.shape)

print("\nMissing values in weather columns:")
weather_columns = [
    "temp", "feelslike", "humidity", "precip",
    "precipprob", "snow", "windgust", "windspeed",
    "sealevelpressure", "cloudcover", "visibility",
    "day", "month", "dayofweek", "weekofyear",
    "is_weekend", "quarter", "dayofyear"
]

print(ride_weather_df[weather_columns].isna().sum().sum())

Shape: (550297, 22)

Missing values in weather columns:
0


In [16]:
combined_df = ride_weather_df.merge(
    zone_df,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

print("combined_df shape:", combined_df.shape)
print("Missing zone values:", combined_df["Zone"].isna().sum())

combined_df shape: (550297, 26)
Missing zone values: 1582


In [17]:
# ==========================================
# Create final modeling dataframe
# ==========================================

model_df = combined_df.copy()

model_df["pickup_date"] = pd.to_datetime(
    model_df["pickup_hour"]
).dt.normalize()

model_df = model_df.sort_values(
    ["pickup_date", "pickup_hour", "PULocationID"]
).reset_index(drop=True)

print("Model dataframe shape:", model_df.shape)

print(
    "Date range:",
    model_df["pickup_date"].min(),
    "to",
    model_df["pickup_date"].max()
)

Model dataframe shape: (550297, 27)
Date range: 2021-04-01 00:00:00 to 2021-06-30 00:00:00


In [18]:
features = [
    "pickup_hour",
    "PULocationID",
    "temp",
    "feelslike",
    "humidity",
    "precip",
    "precipprob",
    "snow",
    "windgust",
    "windspeed",
    "sealevelpressure",
    "cloudcover",
    "visibility",
    "day",
    "month",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "quarter",
    "dayofyear"
]

X = model_df[features]
y = model_df["ride_count"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

X shape: (550297, 20)
y shape: (550297,)
Missing values in X: 0
Missing values in y: 0


In [19]:
# ==========================================
# STEP 3: Time-based train/test split
# ==========================================

train_mask = model_df["pickup_date"] < "2021-06-01"
test_mask = model_df["pickup_date"] >= "2021-06-01"

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining period:")
print(
    model_df.loc[train_mask, "pickup_date"].min(),
    "to",
    model_df.loc[train_mask, "pickup_date"].max()
)

print("\nTesting period:")
print(
    model_df.loc[test_mask, "pickup_date"].min(),
    "to",
    model_df.loc[test_mask, "pickup_date"].max()
)

Training data:
X_train: (368438, 20)
y_train: (368438,)

Testing data:
X_test: (181859, 20)
y_test: (181859,)

Training period:
2021-04-01 00:00:00 to 2021-05-31 00:00:00

Testing period:
2021-06-01 00:00:00 to 2021-06-30 00:00:00


In [20]:
# ==========================================
# Convert pickup_hour to numeric hour
# ==========================================

X_train = X_train.copy()
X_test = X_test.copy()

X_train["pickup_hour"] = X_train["pickup_hour"].dt.hour
X_test["pickup_hour"] = X_test["pickup_hour"].dt.hour

print("pickup_hour dtype:", X_train["pickup_hour"].dtype)

print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

pickup_hour dtype: int32
Missing values in X_train: 0
Missing values in X_test: 0


In [21]:
# ==========================================
# STEP 4: Train Random Forest
# ==========================================

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest model...")

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

Training Random Forest model...
Random Forest training completed!


In [22]:
# ==========================================
# STEP 5: Make predictions
# ==========================================

y_pred = rf_model.predict(X_test)

print("Prediction completed!")
print("Number of predictions:", len(y_pred))

print("\nFirst 10 actual values:")
print(y_test.values[:10])

print("\nFirst 10 predicted values:")
print(y_pred[:10])

Prediction completed!
Number of predictions: 181859

First 10 actual values:
[1 2 1 1 1 2 2 1 1 1]

First 10 predicted values:
[1.26066507 1.26066507 1.26066507 1.26066507 1.26066507 1.26066507
 1.26066507 1.14633333 1.14633333 1.14633333]


In [23]:
print(model_df[["pickup_hour", "PULocationID", "ride_count"]].head(20))


   pickup_hour  PULocationID  ride_count
0   2021-04-01             1           1
1   2021-04-01             1           1
2   2021-04-01             1           2
3   2021-04-01             1           1
4   2021-04-01             1           1
5   2021-04-01             1           1
6   2021-04-01             1           1
7   2021-04-01             1           1
8   2021-04-01             1           1
9   2021-04-01             2           1
10  2021-04-01             3          26
11  2021-04-01             3          15
12  2021-04-01             3          10
13  2021-04-01             3           9
14  2021-04-01             3          11
15  2021-04-01             3          21
16  2021-04-01             3          36
17  2021-04-01             3          64
18  2021-04-01             3          63
19  2021-04-01             3          77


In [24]:
# Recreate model_df without changing the original row order

model_df = combined_df.copy()

model_df["pickup_date"] = pd.to_datetime(
    model_df["pickup_hour"]
).dt.normalize()

print("Model dataframe shape:", model_df.shape)
print("\nFirst 20 rows:")
print(model_df[["pickup_hour", "PULocationID", "ride_count"]].head(20))

Model dataframe shape: (550297, 27)

First 20 rows:
   pickup_hour  PULocationID  ride_count
0   2021-04-01             3          26
1   2021-04-01             4          24
2   2021-04-01             6           5
3   2021-04-01             7          83
4   2021-04-01             9           6
5   2021-04-01            10          34
6   2021-04-01            11           5
7   2021-04-01            12           1
8   2021-04-01            13          23
9   2021-04-01            14          39
10  2021-04-01            15          10
11  2021-04-01            16          11
12  2021-04-01            17         136
13  2021-04-01            18          81
14  2021-04-01            19           8
15  2021-04-01            20          53
16  2021-04-01            21          24
17  2021-04-01            22          25
18  2021-04-01            23          12
19  2021-04-01            24          22


In [25]:
features = [
    "pickup_hour",
    "PULocationID",
    "temp",
    "feelslike",
    "humidity",
    "precip",
    "precipprob",
    "snow",
    "windgust",
    "windspeed",
    "sealevelpressure",
    "cloudcover",
    "visibility",
    "day",
    "month",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "quarter",
    "dayofyear"
]

X = model_df[features]
y = model_df["ride_count"]

train_mask = model_df["pickup_date"] < "2021-06-01"
test_mask = model_df["pickup_date"] >= "2021-06-01"

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()

y_train = y[train_mask]
y_test = y[test_mask]

# Convert datetime to numeric hour
X_train["pickup_hour"] = X_train["pickup_hour"].dt.hour
X_test["pickup_hour"] = X_test["pickup_hour"].dt.hour

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nMissing values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

X_train: (368438, 20)
X_test: (181859, 20)
y_train: (368438,)
y_test: (181859,)

Missing values:
X_train: 0
X_test: 0


In [26]:
#  Train Random Forest

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest model...")

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

Training Random Forest model...
Random Forest training completed!


In [27]:
y_pred = rf_model.predict(X_test)

print("Prediction completed!")
print("Number of predictions:", len(y_pred))

print("\nFirst 10 actual values:")
print(y_test.values[:10])

print("\nFirst 10 predicted values:")
print(y_pred[:10])

Prediction completed!
Number of predictions: 181859

First 10 actual values:
[ 28  54   3   8 163  11  56  10   1  12]

First 10 predicted values:
[ 44.33426185  55.69962354   5.40280986  10.51746978 167.0789618
  15.32949694  69.60204187  20.63963734   4.49051951  87.34181855]


In [28]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.3f}")

RMSE: 51.50
MAE:  31.19
R²:   0.619


In [29]:
import numpy as np

for df_ in [X_train, X_test]:
    df_["hour_sin"] = np.sin(2 * np.pi * df_["pickup_hour"] / 24)
    df_["hour_cos"] = np.cos(2 * np.pi * df_["pickup_hour"] / 24)
    df_["dow_sin"] = np.sin(2 * np.pi * df_["dayofweek"] / 7)
    df_["dow_cos"] = np.cos(2 * np.pi * df_["dayofweek"] / 7)

In [30]:

# Create lag features

model_df = model_df.sort_values(
    ["PULocationID", "pickup_hour"]
).copy()

for lag in [1, 24, 168]:
    model_df[f"lag_{lag}"] = (
        model_df.groupby("PULocationID")["ride_count"].shift(lag)
    )

# Check the new features
print(model_df[[
    "PULocationID",
    "pickup_hour",
    "ride_count",
    "lag_1",
    "lag_24",
    "lag_168"
]].head(20))

       PULocationID pickup_hour  ride_count  lag_1  lag_24  lag_168
977               1  2021-04-01           1    NaN     NaN      NaN
1716              1  2021-04-01           1    1.0     NaN      NaN
3237              1  2021-04-01           2    1.0     NaN      NaN
3742              1  2021-04-01           1    2.0     NaN      NaN
3998              1  2021-04-01           1    1.0     NaN      NaN
4250              1  2021-04-01           1    1.0     NaN      NaN
4506              1  2021-04-01           1    1.0     NaN      NaN
4762              1  2021-04-01           1    1.0     NaN      NaN
5018              1  2021-04-01           1    1.0     NaN      NaN
6531              1  2021-04-02           1    1.0     NaN      NaN
8511              1  2021-04-02           2    1.0     NaN      NaN
9276              1  2021-04-02           2    2.0     NaN      NaN
10549             1  2021-04-02           2    2.0     NaN      NaN
10803             1  2021-04-02           1    2

In [31]:
duplicates = model_df.duplicated(
    subset=["PULocationID", "pickup_hour"]
).sum()

print("Duplicate location-hour rows:", duplicates)

Duplicate location-hour rows: 526682


In [32]:
print(
    trip_df.groupby(["pickup_hour", "PULocationID"])
    .size()
    .value_counts()
    .sort_index()
    .head(20)
)

1      67
2      39
3      31
4      24
5      19
6      29
7      33
8      41
9      41
10     45
11     56
12     69
13     75
14     78
15     80
16     88
17     95
18    123
19     94
20    132
Name: count, dtype: int64


In [33]:
# Reload/rebuild trip_df with pickup_hour untouched (keep the actual hour!)
trip_df = pd.concat([april, may, june], ignore_index=True)
trip_df["pickup_hour"] = pd.to_datetime(trip_df["pickup_hour"])  # no normalize here

# Separate column just for matching to daily weather
trip_df["pickup_date"] = trip_df["pickup_hour"].dt.normalize()
weather_df["pickup_date"] = pd.to_datetime(weather_df["datetime"]).dt.normalize()

ride_weather_df = trip_df.merge(
    weather_df,
    on="pickup_date",   # join on date, not on the real hourly timestamp
    how="left"
)

print("Shape:", ride_weather_df.shape)
print("Duplicate (location, hour) rows:",
      ride_weather_df.duplicated(subset=["PULocationID", "pickup_hour"]).sum())

Shape: (550297, 23)
Duplicate (location, hour) rows: 0


In [35]:
# Rebuild trip_df WITHOUT normalizing pickup_hour

trip_df = pd.concat([april, may, june], ignore_index=True)
trip_df["pickup_hour"] = pd.to_datetime(trip_df["pickup_hour"])  # keep real hour, no normalize

# separate column just for joining to daily weather
trip_df["pickup_date"] = trip_df["pickup_hour"].dt.normalize()

In [38]:
#Merge weather on pickup_date (not pickup_hour)
weather_df["pickup_date"] = pd.to_datetime(weather_df["datetime"]).dt.normalize()

ride_weather_df = trip_df.merge(
    weather_df,
    on="pickup_date",
    how="left"
)

print("ride_weather_df shape:", ride_weather_df.shape)

# sanity check: this should now be 0
dupe_check = ride_weather_df.duplicated(subset=["PULocationID", "pickup_hour"]).sum()
print("Duplicate (location, hour) rows:", dupe_check)

ride_weather_df shape: (550297, 23)
Duplicate (location, hour) rows: 0


In [39]:
#Merge zone lookup

combined_df = ride_weather_df.merge(
    zone_df,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

print("combined_df shape:", combined_df.shape)
print("Missing zone values:", combined_df["Zone"].isna().sum())


combined_df shape: (550297, 27)
Missing zone values: 1582


In [40]:
missing_ids = combined_df.loc[combined_df["Zone"].isna(), "PULocationID"].unique()
print(sorted(missing_ids))

[np.int64(265)]


In [41]:
model_df = combined_df.copy()
model_df["hour_of_day"] = model_df["pickup_hour"].dt.hour

model_df = model_df[~model_df["PULocationID"].isin(missing_ids)]  # drop unknown zones

model_df = model_df.sort_values(
    ["PULocationID", "pickup_hour"]
).reset_index(drop=True)

print(model_df.shape)
print(model_df[["pickup_hour", "hour_of_day", "PULocationID", "ride_count"]].head(10))

(548715, 28)
          pickup_hour  hour_of_day  PULocationID  ride_count
0 2021-04-01 04:00:00            4             1           1
1 2021-04-01 07:00:00            7             1           1
2 2021-04-01 13:00:00           13             1           2
3 2021-04-01 15:00:00           15             1           1
4 2021-04-01 16:00:00           16             1           1
5 2021-04-01 17:00:00           17             1           1
6 2021-04-01 18:00:00           18             1           1
7 2021-04-01 19:00:00           19             1           1
8 2021-04-01 20:00:00           20             1           1
9 2021-04-02 02:00:00            2             1           1


In [42]:
#add lagging features
model_df = model_df.sort_values(["PULocationID", "pickup_hour"]).reset_index(drop=True)

for lag in [1, 24, 168]:
    model_df[f"lag_{lag}"] = (
        model_df.groupby("PULocationID")["ride_count"].shift(lag)
    )

print(model_df[[
    "PULocationID", "pickup_hour", "ride_count",
    "lag_1", "lag_24", "lag_168"
]].head(20))

    PULocationID         pickup_hour  ride_count  lag_1  lag_24  lag_168
0              1 2021-04-01 04:00:00           1    NaN     NaN      NaN
1              1 2021-04-01 07:00:00           1    1.0     NaN      NaN
2              1 2021-04-01 13:00:00           2    1.0     NaN      NaN
3              1 2021-04-01 15:00:00           1    2.0     NaN      NaN
4              1 2021-04-01 16:00:00           1    1.0     NaN      NaN
5              1 2021-04-01 17:00:00           1    1.0     NaN      NaN
6              1 2021-04-01 18:00:00           1    1.0     NaN      NaN
7              1 2021-04-01 19:00:00           1    1.0     NaN      NaN
8              1 2021-04-01 20:00:00           1    1.0     NaN      NaN
9              1 2021-04-02 02:00:00           1    1.0     NaN      NaN
10             1 2021-04-02 10:00:00           2    1.0     NaN      NaN
11             1 2021-04-02 13:00:00           2    2.0     NaN      NaN
12             1 2021-04-02 18:00:00           2   

In [43]:
# Rolling mean feature


model_df["rolling_mean_24"] = (
    model_df.groupby("PULocationID")["ride_count"]
    .transform(lambda s: s.shift(1).rolling(24).mean())
)

In [44]:
# Drop rows missing lag/rolling history

print("Rows before dropping NaN history rows:", len(model_df))
model_df = model_df.dropna(
    subset=["lag_1", "lag_24", "lag_168", "rolling_mean_24"]
)
print("Rows after:", len(model_df))

Rows before dropping NaN history rows: 548715
Rows after: 505284


In [45]:
# Cyclical encoding for hour and day-of-week

model_df["hour_sin"] = np.sin(2 * np.pi * model_df["hour_of_day"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["hour_of_day"] / 24)
model_df["dow_sin"] = np.sin(2 * np.pi * model_df["dayofweek"] / 7)
model_df["dow_cos"] = np.cos(2 * np.pi * model_df["dayofweek"] / 7)

In [46]:
# Updated feature list

features = [
    "hour_of_day", "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "PULocationID",
    "temp", "feelslike", "humidity", "precip", "precipprob",
    "snow", "windgust", "windspeed", "sealevelpressure",
    "cloudcover", "visibility",
    "day", "month", "weekofyear", "is_weekend", "quarter", "dayofyear",
    "lag_1", "lag_24", "lag_168", "rolling_mean_24"
]

X = model_df[features]
y = model_df["ride_count"]

print("X shape:", X.shape)
print("Missing values in X:", X.isna().sum().sum())

X shape: (505284, 27)
Missing values in X: 0


In [47]:
#Time-based train/test split (redo on cleaned model_df)

train_mask = model_df["pickup_date"] < "2021-06-01"
test_mask = model_df["pickup_date"] >= "2021-06-01"

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y[train_mask]
y_test = y[test_mask]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)


X_train: (324063, 27) | X_test: (181221, 27)


In [48]:
#  Retrain Random Forest

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

In [49]:
# Evaluate and compare to baseline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}  (baseline was 51.50)")
print(f"MAE:  {mae:.2f}  (baseline was 31.19)")
print(f"R²:   {r2:.3f}  (baseline was 0.619)")

RMSE: 18.42  (baseline was 51.50)
MAE:  10.60  (baseline was 31.19)
R²:   0.951  (baseline was 0.619)


In [50]:
#checking feature importance
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

lag_168             0.854612
lag_1               0.101938
lag_24              0.012422
rolling_mean_24     0.005375
PULocationID        0.003380
hour_of_day         0.003264
hour_cos            0.002538
hour_sin            0.002116
dayofyear           0.001455
sealevelpressure    0.001417
dtype: float64


In [51]:
#comparing against naive baseline
naive_pred = X_test["lag_1"]

naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
naive_mae = mean_absolute_error(y_test, naive_pred)
naive_r2 = r2_score(y_test, naive_pred)

print(f"Naive baseline (predict = lag_1):")
print(f"RMSE: {naive_rmse:.2f}")
print(f"MAE:  {naive_mae:.2f}")
print(f"R²:   {naive_r2:.3f}")

print(f"\nYour RF model beats naive by:")
print(f"RMSE improvement: {naive_rmse - rmse:.2f}")
print(f"R² improvement:   {r2 - naive_r2:.3f}")

Naive baseline (predict = lag_1):
RMSE: 25.35
MAE:  15.48
R²:   0.908

Your RF model beats naive by:
RMSE improvement: 6.93
R² improvement:   0.044


In [52]:
naive_168 = X_test["lag_168"]

naive_168_rmse = np.sqrt(mean_squared_error(y_test, naive_168))
naive_168_r2 = r2_score(y_test, naive_168)

print(f"Naive baseline (predict = lag_168):")
print(f"RMSE: {naive_168_rmse:.2f}")
print(f"R²:   {naive_168_r2:.3f}")

Naive baseline (predict = lag_168):
RMSE: 23.73
R²:   0.919
